# Canonical Model 01 · Packages and Observations

Inspect the package topology and the canonical **observation** layer. The same
named targets that drive the plots in 02 are the exact targets PEST calibrates
in 04–06 — one definition, reused everywhere.

> **Observation principle:** targets belong to the model. Heads capture the
> infiltration-pond mound, the regional center, and both pumping responses; DRN
> targets aggregate the north and south **seepage** spring slopes; LAK and SFR
> targets read real MF6 observation output for lake stage, stream stage, and
> routed flow.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header('01', 'Packages and Observations', 'Named physical features become reusable observations.')

model = mf.build_canonical_model(Path('../artifacts/canonical_observations') / 'gwf',
                                 config=mf.CanonicalModelConfig.validation())
success, report = model.run_simulation()
assert success, '\n'.join(report[-30:])

print('packages :', sorted(str(n).lower() for n in model.gwf.package_names))
print('targets  :', list(model.targets.keys()))

## The observation network

Each target is a small table indexed by time (or stress period). They are the
model's *measurable* outputs:

- **heads** — `pond_mound`, `regional_center`, `shallow_pumping`, `deep_pumping`
- **drn_flow** — `north_springs`, `south_springs` (aquifer **seepage**)
- **lake_stage** — `valley_lake`
- **sfr_stage** — `sfr_upstream`, `sfr_midpoint`
- **sfr_flow** — `sfr_midflow`, `sfr_outflow` (routed discharge toward the lake)

In [ ]:
from IPython.display import display

display(model.targets.heads.simulated_heads())
display(model.targets.drn_flow.simulated_series())
display(model.targets.lake_stage.simulated_series())
display(model.targets.sfr_stage.simulated_series())
display(model.targets.sfr_flow.simulated_series())

# Compact magnitudes: head change at each well and peak seepage at each slope.
mf.canonical_feature_signals(model)

## Plot the signals

**What to look for:** the pond-mound head rises seasonally without overwhelming
the regional pattern; both pumping wells draw their heads down; and the two
seepage slopes discharge with *different* time series (they tap different
aquifers). These transient signals are what the calibration must reproduce.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

heads = model.targets.heads.simulated_heads()
seep = model.targets.drn_flow.simulated_series()

with sns.axes_style('whitegrid'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for col in [c for c in heads.columns if c != 'per']:
    ax1.plot(heads['per'], heads[col], marker='o', label=col)
ax1.set_xlabel('stress period'); ax1.set_ylabel('head (ft)')
ax1.set_title('Observation heads'); ax1.legend(fontsize=8)
for col in [c for c in seep.columns if c != 'time']:
    ax2.plot(seep['time'], seep[col], marker='s', label=col)
ax2.set_xlabel('time (days)'); ax2.set_ylabel('drain seepage (ft$^3$/day)')
ax2.set_title('Seepage at the spring slopes'); ax2.legend(fontsize=8)
fig.tight_layout()

## Questions worth asking

- Does the infiltration-pond head rise without dominating the regional gradient?
- Do both seepage slopes discharge, and do their series differ between aquifers?
- Are lake and stream observations real MF6 observation output (not post-hoc)?
- Can every target pass straight into a calibration plot and PEST setup?

Continue to **02 · Visual Diagnostics** to see why these signals matter spatially.